# ComfyUI (GGUF) ? Qwen-Image-Edit-2511 (Colab)

This notebook:
- installs ComfyUI + ComfyUI-Manager + ComfyUI-GGUF
- downloads Qwen-Image-Edit-2511 GGUF, Qwen2.5-VL GGUF, VAE, mmproj, and 8-step LoRA
- launches ComfyUI via Cloudflare Tunnel

Quick start:
1. Run all cells from top to bottom.
2. Open the tunnel URL and load your workflow.
3. If VRAM is tight, switch to smaller GGUF quants.


## 1) Installation
Install ComfyUI, ComfyUI-Manager, ComfyUI-GGUF, and swap for better Colab stability.


In [ ]:
# @title 1) Tokens (Colab Secrets / environment only)
# Tokens are read from Colab Secrets (panel 🔑) or process environment.
# Handles late "Grant Access": if secret not found immediately we poll up to ~90s
# so user can enable Notebook access ON + Grant Access during first cell.
import os, time

!pip install -q -U huggingface_hub
from huggingface_hub import login

try:
    from google.colab import userdata
except Exception:
    userdata = None


def colab_secret(name: str) -> str:
    """Read a secret from env first, then Colab Secrets."""
    value = os.environ.get(name, "").strip()
    if not value and userdata is not None:
        try:
            raw = userdata.get(name) or ""
        except Exception:
            raw = ""
        if isinstance(raw, dict):
            raw = raw.get("value") or raw.get("token") or next(iter(raw.values()), "")
        value = str(raw).strip()
    return value


HF_TOKEN = colab_secret("HF_TOKEN") or colab_secret("HUGGINGFACE_TOKEN")
if not HF_TOKEN:
    print("HF_TOKEN не найден — жду до 90 сек (можешь сейчас нажать Grant Access / включить Notebook access ON)...")
    for _attempt in range(30):
        time.sleep(3)
        HF_TOKEN = colab_secret("HF_TOKEN") or colab_secret("HUGGINGFACE_TOKEN")
        if HF_TOKEN:
            print("✓ HF_TOKEN появился — продолжаю.")
            break
        print(".", end="", flush=True)
    print()
if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found.\n"
        "  • Colab: открой 🔑 Secrets слева, добавь HF_TOKEN, включи Notebook access ON, нажми Grant Access, затем Runtime → Rerun.\n"
        "  • Если ты нажал Grant Access только что — токен должен был подхватиться за 90 сек; если не подхватился, перезапусти ячейку.\n"
        "  • Jupyter локально: export HF_TOKEN=... перед стартом."
    )
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
print("HF auth: OK")

CIVITAI_API_TOKEN = colab_secret("CIVITAI_API_TOKEN")
if CIVITAI_API_TOKEN:
    os.environ["CIVITAI_API_TOKEN"] = CIVITAI_API_TOKEN
    print("Civitai auth: OK")
else:
    print("Civitai token not set (optional); Civitai downloads may return 403.")

print("Done.")


In [ ]:
# @title 2) Install ComfyUI + Managers + node pack (incl. rgthree)
# Installs ComfyUI, ComfyUI-Manager, ComfyUI-Model-Manager, ComfyUI-GGUF,
# KJNodes and rgthree-comfy. Idempotent: safe to rerun.
import os
import subprocess
import sys


def sh(cmd):
    print("+", cmd)
    subprocess.run(cmd, shell=True, check=True)


def pip_ok(spec):
    rc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", spec], check=False
    ).returncode
    return rc == 0


# Swap protects small-RAM Colab VMs during dependency resolution.
if not os.path.exists("/swapfile"):
    print("Creating swap (8GB)...")
    !sudo fallocate -l 8G /swapfile
    !sudo chmod 600 /swapfile
    !sudo mkswap /swapfile
    _swap_result = subprocess.run(
        ["sudo", "swapon", "/swapfile"], capture_output=True, text=True
    )
    if _swap_result.returncode != 0:
        print("Swap unavailable on this Colab VM; continuing without swap.")
    else:
        print("Swap enabled.")

!apt-get -y update -qq
!apt-get -y install -qq aria2 ffmpeg || true
!apt-get -y install -qq cloudflared 2>/dev/null || echo "cloudflared apt not available"

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

COMFY_ROOT = "/content/ComfyUI"

# ── ComfyUI core ──────────────────────────────────────────────────────────────
if not os.path.exists(COMFY_ROOT):
    print("Cloning ComfyUI...")
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {COMFY_ROOT}

print("Installing ComfyUI requirements...")
!pip install -q -U pip
!pip install -q -r {COMFY_ROOT}/requirements.txt

# ── Optional attention accelerators (GPU-aware, never fatal) ────────────────
ENABLE_ACCEL_PACK = True
if ENABLE_ACCEL_PACK:
    try:
        import torch

        gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
        cc_major = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
        print(f"GPU detected: {gpu_name} (sm{cc_major}0)")
        print("xformers:", "OK" if pip_ok("xformers>=0.0.28.post3") else "FAILED (continuing)")
        if cc_major >= 8:
            print("flash-attn:", "OK" if pip_ok("flash-attn>=2.7.0.post2") else "FAILED (continuing)")
            print("sageattention:", "OK" if pip_ok("sageattention>=2.1.1") else "FAILED (continuing)")
        else:
            print("Skipping flash-attn/sageattention on sm<8 (T4-safe xformers path).")
    except Exception as exc:
        print("Accelerator pack skipped:", exc)

# ── Custom nodes ──────────────────────────────────────────────────────────────
NODES = {
    "ComfyUI-Manager": "https://github.com/ltdrdata/ComfyUI-Manager.git",
    # Separate model manager (different project from ComfyUI-Manager):
    # Pinned to v2.8.4: the v2.8.5 tag ships no dist.tar.gz release asset yet,
    # which makes its web UI download fail on every startup.
    "ComfyUI-Model-Manager": "https://github.com/hayden-cn/ComfyUI-Model-Manager.git#v2.8.4",
    "ComfyUI-GGUF": "https://github.com/city96/ComfyUI-GGUF.git",
    "ComfyUI-Impact-Pack": "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git",
    "ComfyUI-Impact-Subpack": "https://github.com/ltdrdata/ComfyUI-Impact-Subpack.git",
    "comfyui-kjnodes": "https://github.com/kijai/ComfyUI-KJNodes.git",
    "rgthree-comfy": "https://github.com/rgthree/rgthree-comfy.git",
    "seedvr2_videoupscaler": "https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git",
    "ComfyUI-Workflow-Models-Downloader": "https://github.com/slahiri/ComfyUI-Workflow-Models-Downloader.git",
}
for folder, repo in NODES.items():
    target = f"{COMFY_ROOT}/custom_nodes/{folder}"
    if not os.path.exists(target):
        print("Cloning", folder, "...")
        repo_url, _, ref = repo.partition("#")
        clone_cmd = ["git", "clone", "--depth", "1"]
        if ref:
            # Works for both branches and tags (unlike fetch+checkout on a
            # shallow clone, which cannot resolve tags).
            clone_cmd += ["--branch", ref]
        clone_cmd += [repo_url, target]
        subprocess.run(clone_cmd, check=True)
    req = f"{target}/requirements.txt"
    if os.path.exists(req):
        !pip install -q -r {req}

print("\nInstalled node packs:")
for folder in sorted(os.listdir(f"{COMFY_ROOT}/custom_nodes")):
    if not folder.startswith("__"):
        print("  -", folder)
print("\nComfyUI + Managers + node pack ready.")


## 2) Quant Settings
Choose GGUF quant defaults and review rough file-size/VRAM estimates.


In [ ]:
# @title 2) Settings: GGUF quant selection + size/VRAM table
import math

# ---- Auto quant by VRAM ----
AUTO_QUANT_BY_VRAM = True
AUTO_QUANT_VRAM_FRACTION = 0.7
AUTO_VRAM_FALLBACK_GB = 14.0


# ---- Quant selection (editable) ----
QWEN_EDIT_QUANT = 'Q2_K'   # options: Q2_K, Q3_K_S, Q3_K_M, Q3_K_L, Q4_0, Q4_1, Q4_K_S, Q4_K_M, Q5_0, Q5_1, Q5_K_S, Q5_K_M, Q6_K, Q8_0, BF16, F16

# Text encoder (GGUF)
QWEN_VL_QUANT = 'Q4_K_M'  # options: Q4_K_M, Q8_0, F16

# mmproj for Qwen2.5-VL
DOWNLOAD_MMPROJ = True
MMPROJ_QUANT = 'Q8_0'     # options: Q8_0, f16

# LoRA (8 steps)
DOWNLOAD_LORA = True
LORA_FILE = 'Qwen-Image-Edit-2511-Lightning-8steps-V1.0-bf16.safetensors'

# If VRAM is limited, keep this enabled to offload text encoder to CPU/RAM
OFFLOAD_TEXT_ENCODER = True

# ---- Size tables (GB) from Hugging Face listings ----
# Qwen-Image-Edit-2511 GGUF (unsloth)
QWEN_EDIT_SIZES_GB = {
  'Q2_K': 7.47,
  'Q3_K_S': 9.22,
  'Q3_K_M': 9.92,
  'Q3_K_L': 10.6,
  'Q4_0': 11.9,
  'Q4_1': 12.8,
  'Q4_K_S': 12.4,
  'Q4_K_M': 13.2,
  'Q5_0': 14.4,
  'Q5_1': 15.4,
  'Q5_K_S': 14.3,
  'Q5_K_M': 15.0,
  'Q6_K': 16.9,
  'Q8_0': 21.8,
  'BF16': 40.9,
  'F16': 40.9
}

# Qwen2.5-VL-7B-Instruct GGUF (ggml-org)
QWEN_VL_SIZES_GB = {
  'Q4_K_M': 4.68,
  'Q8_0': 8.10,
  'F16': 15.2
}

# mmproj sizes (GB)
MMPROJ_SIZES_GB = {
  'Q8_0': 0.853,
  'f16': 1.35
}

# VAE size (GB)
QWEN_EDIT_VAE_GB = 0.254  # official Comfy-Org qwen_image_vae.safetensors

# LoRA size (GB)
LORA_GB = 0.85  # Qwen-Image-Edit-Lightning-8steps-V1.0-bf16




def detect_total_vram_gb(default=AUTO_VRAM_FALLBACK_GB):
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.get_device_properties(0).total_memory / (1024**3)
    except Exception as e:
        print(f"VRAM detection failed ({e}); using fallback {default:.2f} GB")
    return default


def _is_supported_quant_name(qname):
    q = str(qname).upper().strip()
    if q.startswith('UD-') or q.startswith('IQ'):
        return False
    if q in {'BF16', 'F16', 'F32'}:
        return True
    return q.startswith('Q')

def _supported_quant_items(size_dict):
    return sorted([(k, v) for k, v in size_dict.items() if _is_supported_quant_name(k)], key=lambda kv: kv[1])

def pick_best_quant(size_dict, budget_gb, overhead=1.10):
    items = _supported_quant_items(size_dict)
    if not items:
        items = sorted(size_dict.items(), key=lambda kv: kv[1])
    fitting = [(k, v) for k, v in items if (v * overhead) <= budget_gb]
    if fitting:
        return fitting[-1][0]
    return items[0][0]


def est_vram_gb(size_gb, overhead=1.10):
    return size_gb * overhead


def print_table(title, d):
    print('\n' + title)
    print('-' * len(title))
    for k, v in d.items():
        print(f"{k:8s}  file~{v:5.2f} GB   VRAM~{est_vram_gb(v):5.2f} GB")


# ---- Auto apply quant choice based on detected VRAM ----
TOTAL_VRAM_GB = detect_total_vram_gb()
AUTO_BUDGET_GB = TOTAL_VRAM_GB * AUTO_QUANT_VRAM_FRACTION
print(f"\nGPU VRAM detected: ~{TOTAL_VRAM_GB:.2f} GB | Auto budget (75%): ~{AUTO_BUDGET_GB:.2f} GB")

if AUTO_QUANT_BY_VRAM:
    lora_side = est_vram_gb(LORA_GB) if DOWNLOAD_LORA else 0.0

    if OFFLOAD_TEXT_ENCODER:
        budget_edit = max(AUTO_BUDGET_GB - est_vram_gb(QWEN_EDIT_VAE_GB) - lora_side, 0.01)
        QWEN_EDIT_QUANT = pick_best_quant(QWEN_EDIT_SIZES_GB, budget_edit)
    else:
        if DOWNLOAD_MMPROJ:
            best_fit = None
            best_any = None
            for e_k, e_v in _supported_quant_items(QWEN_EDIT_SIZES_GB):
                for v_k, v_v in _supported_quant_items(QWEN_VL_SIZES_GB):
                    for m_k, m_v in _supported_quant_items(MMPROJ_SIZES_GB):
                        total = est_vram_gb(e_v) + est_vram_gb(v_v) + est_vram_gb(m_v) + est_vram_gb(QWEN_EDIT_VAE_GB) + lora_side
                        score = (e_v, v_v, m_v)
                        if (best_any is None) or (total < best_any[0]):
                            best_any = (total, e_k, v_k, m_k)
                        if total <= AUTO_BUDGET_GB:
                            if (best_fit is None) or (score > best_fit[0]) or (score == best_fit[0] and total < best_fit[1]):
                                best_fit = (score, total, e_k, v_k, m_k)
            if best_fit is not None:
                QWEN_EDIT_QUANT, QWEN_VL_QUANT, MMPROJ_QUANT = best_fit[2], best_fit[3], best_fit[4]
            else:
                QWEN_EDIT_QUANT, QWEN_VL_QUANT, MMPROJ_QUANT = best_any[1], best_any[2], best_any[3]
        else:
            best_fit = None
            best_any = None
            for e_k, e_v in _supported_quant_items(QWEN_EDIT_SIZES_GB):
                for v_k, v_v in _supported_quant_items(QWEN_VL_SIZES_GB):
                    total = est_vram_gb(e_v) + est_vram_gb(v_v) + est_vram_gb(QWEN_EDIT_VAE_GB) + lora_side
                    score = (e_v, v_v)
                    if (best_any is None) or (total < best_any[0]):
                        best_any = (total, e_k, v_k)
                    if total <= AUTO_BUDGET_GB:
                        if (best_fit is None) or (score > best_fit[0]) or (score == best_fit[0] and total < best_fit[1]):
                            best_fit = (score, total, e_k, v_k)
            if best_fit is not None:
                QWEN_EDIT_QUANT, QWEN_VL_QUANT = best_fit[2], best_fit[3]
            else:
                QWEN_EDIT_QUANT, QWEN_VL_QUANT = best_any[1], best_any[2]

    print(f"Auto quant active: QWEN_EDIT_QUANT={QWEN_EDIT_QUANT}, QWEN_VL_QUANT={QWEN_VL_QUANT}, MMPROJ_QUANT={MMPROJ_QUANT}")

print_table('Qwen-Image-Edit-2511 GGUF (DiT) quants', QWEN_EDIT_SIZES_GB)
print_table('Qwen2.5-VL-7B-Instruct GGUF (text encoder) quants', QWEN_VL_SIZES_GB)
print_table('mmproj GGUF', MMPROJ_SIZES_GB)

print('\nSelected:')
print('  QWEN_EDIT_QUANT     =', QWEN_EDIT_QUANT)
print('  QWEN_VL_QUANT       =', QWEN_VL_QUANT)
print('  DOWNLOAD_MMPROJ     =', DOWNLOAD_MMPROJ, 'MMPROJ_QUANT =', MMPROJ_QUANT)
print('  DOWNLOAD_LORA       =', DOWNLOAD_LORA, 'LORA_FILE =', LORA_FILE)
print('  OFFLOAD_TEXT_ENCODER =', OFFLOAD_TEXT_ENCODER)

# Rough total (weights only, excludes activations and current resolution)
total = 0.0
total += est_vram_gb(QWEN_EDIT_SIZES_GB[QWEN_EDIT_QUANT])
total += est_vram_gb(QWEN_EDIT_VAE_GB)
if DOWNLOAD_LORA:
    total += est_vram_gb(LORA_GB)
if not OFFLOAD_TEXT_ENCODER:
    total += est_vram_gb(QWEN_VL_SIZES_GB[QWEN_VL_QUANT])
    if DOWNLOAD_MMPROJ:
        total += est_vram_gb(MMPROJ_SIZES_GB[MMPROJ_QUANT])

print(f"\nRough VRAM for weights if all kept on GPU at once: ~{total:.2f} GB")
print("Note: real peak VRAM depends on resolution/batch/latents; --lowvram/offload is recommended for T4 10GB.")


# ---- RAM-aware auto guard (Colab-safe) ----
AUTO_QUANT_BY_RAM = True
AUTO_QUANT_RAM_FRACTION = 0.6
AUTO_RAM_OS_RESERVE_GB = 2.0
AUTO_RAM_RUNTIME_GB = 1.2
AUTO_RAM_SPILL_FACTOR = 0.45
AUTO_RAM_GPU_RESIDENT_FACTOR = 0.10
AUTO_RAM_TEXT_OFFLOAD_FACTOR = 1.05
RAM_GUARD_ASSUME_SINGLE_MODEL_RUN = True


def detect_total_ram_gb(default=12.9):
    try:
        import psutil
        return psutil.virtual_memory().total / (1024**3)
    except Exception as e:
        print(f"RAM detection failed ({e}); using fallback {default:.2f} GB")
    return default


def _norm_tokens(name):
    return [t for t in str(name).upper().replace('-', '_').split('_') if t]


def _is_text_quant_var(qvar):
    toks = _norm_tokens(qvar)
    text_keys = {'T5', 'QWEN', 'UMT5', 'FLAN', 'CLIP', 'TEXT', 'ENCODER', 'TE'}
    return any((t in text_keys) or t.startswith('QWEN') or t.startswith('T5') for t in toks)


def _size_dict_for_quant_var(qvar, g):
    pref = qvar[:-6] if qvar.endswith('_QUANT') else qvar
    candidates = [
        f'{pref}_SIZES_GB',
        f'{pref}_SIZE_GB',
        f'{pref}_GGUF_GB',
    ]
    for c in candidates:
        if c in g and isinstance(g[c], dict):
            return c
    # fallback: first dict with matching prefix
    for k, v in g.items():
        if isinstance(v, dict) and k.endswith(('_SIZES_GB', '_GGUF_GB')) and pref in k:
            return k
    return None


def _smallest_quant_key(size_dict):
    if not size_dict:
        return None
    items = _supported_quant_items(size_dict)
    if items:
        return items[0][0]
    return sorted(size_dict.items(), key=lambda kv: kv[1])[0][0]

MIN_QUANT_FLOORS = {
    'T5': 'Q5_K_M',
    'QWEN': 'Q4_K_S',
    'SRPO': 'Q5_K',
    'KLEIN': 'Q5_K_M',
    'ZIMAGE': 'Q5_K_M',
    'CHROMA': 'Q5_K_M',
    'WAN': 'Q5_K_M',
    'LTX': 'Q5_K_M',
}

def _floor_quant_key(qvar, size_dict):
    q = str(qvar).upper()
    for family, floor in MIN_QUANT_FLOORS.items():
        if family in q and floor in size_dict:
            return floor
    return _smallest_quant_key(size_dict)

def _reduce_quant_with_floor(qvar, current, size_dict):
    if not isinstance(size_dict, dict) or not size_dict or current not in size_dict:
        return current

    floor = _floor_quant_key(qvar, size_dict)
    if floor not in size_dict:
        floor = _smallest_quant_key(size_dict)

    cur_size = float(size_dict[current])
    floor_size = float(size_dict[floor])

    if cur_size <= floor_size:
        return current

    candidates = sorted((k, float(v)) for k, v in size_dict.items())
    step_down = [k for k, s in candidates if floor_size <= s < cur_size]
    if step_down:
        return step_down[-1]
    return floor


def _estimate_ram_peak_gb(g):
    # Selected quant sizes
    quant_sizes = {}
    for k, v in list(g.items()):
        if not (isinstance(k, str) and k.endswith('_QUANT') and isinstance(v, str)):
            continue
        dname = _size_dict_for_quant_var(k, g)
        if not dname:
            continue
        d = g[dname]
        if v in d:
            quant_sizes[k] = float(d[v])

    # Fixed model components from scalar *_GB constants
    fixed_components = []
    for k, v in list(g.items()):
        if not (isinstance(k, str) and k.endswith('_GB') and isinstance(v, (int, float))):
            continue
        if k.startswith('AUTO_'):
            continue
        if any(x in k for x in ['VRAM', 'RAM', 'BUDGET', 'FALLBACK', 'FRACTION', 'RESERVE', 'RUNTIME', 'SPILL', 'RESIDENT']):
            continue
        fixed_components.append(float(v))

    offload = bool(g.get('OFFLOAD_TEXT_ENCODER', g.get('OFFLOAD_TEXT_ENCODERS', False)))

    text_vals = []
    core_vals = []
    for qvar, size in quant_sizes.items():
        if _is_text_quant_var(qvar):
            text_vals.append(float(size))
        else:
            core_vals.append(float(size))

    if RAM_GUARD_ASSUME_SINGLE_MODEL_RUN:
        text_gb = max(text_vals) if text_vals else 0.0
        core_gb = max(core_vals) if core_vals else 0.0
    else:
        text_gb = sum(text_vals)
        core_gb = sum(core_vals)

    fixed_gb = sum(fixed_components)
    resident_proxy = core_gb + fixed_gb

    text_ram = text_gb * (AUTO_RAM_TEXT_OFFLOAD_FACTOR if offload else 0.20)
    spill_ram = resident_proxy * AUTO_RAM_SPILL_FACTOR
    resident_ram = resident_proxy * AUTO_RAM_GPU_RESIDENT_FACTOR

    ram_peak = AUTO_RAM_OS_RESERVE_GB + AUTO_RAM_RUNTIME_GB + text_ram + spill_ram + resident_ram
    return ram_peak, quant_sizes, text_gb, core_gb, fixed_gb


TOTAL_RAM_GB = detect_total_ram_gb()
RAM_BUDGET_GB = TOTAL_RAM_GB * AUTO_QUANT_RAM_FRACTION
RAM_PEAK_GB, _ram_quant_sizes, _ram_text_gb, _ram_core_gb, _ram_fixed_gb = _estimate_ram_peak_gb(globals())

print(f"\nRAM detected: ~{TOTAL_RAM_GB:.2f} GB | RAM budget ({int(AUTO_QUANT_RAM_FRACTION*100)}%): ~{RAM_BUDGET_GB:.2f} GB")
print(f"Estimated RAM peak: ~{RAM_PEAK_GB:.2f} GB (text~{_ram_text_gb:.2f}, core~{_ram_core_gb:.2f}, fixed~{_ram_fixed_gb:.2f})")

if AUTO_QUANT_BY_RAM and RAM_PEAK_GB > RAM_BUDGET_GB:
    print("RAM guard: estimated peak exceeds budget. Trying safer quant fallback with quality floors...")

    # 1) shrink text quants first
    for qvar in sorted(list(globals().keys())):
        if not (qvar.endswith('_QUANT') and _is_text_quant_var(qvar)):
            continue
        dname = _size_dict_for_quant_var(qvar, globals())
        if not dname:
            continue
        d = globals()[dname]
        if isinstance(d, dict) and d:
                current = globals().get(qvar)
                reduced = _reduce_quant_with_floor(qvar, current, d)
                if reduced is not None and current != reduced:
                    print(f"  RAM guard: {qvar} {current} -> {reduced}")
                    globals()[qvar] = reduced

    RAM_PEAK_GB, _, _, _, _ = _estimate_ram_peak_gb(globals())

    # 2) if still high, shrink remaining quants
    if RAM_PEAK_GB > RAM_BUDGET_GB:
        for qvar in sorted(list(globals().keys())):
            if not qvar.endswith('_QUANT'):
                continue
            if _is_text_quant_var(qvar):
                continue
            dname = _size_dict_for_quant_var(qvar, globals())
            if not dname:
                continue
            d = globals()[dname]
            if isinstance(d, dict) and d:
                current = globals().get(qvar)
                reduced = _reduce_quant_with_floor(qvar, current, d)
                if reduced is not None and current != reduced:
                    print(f"  RAM guard: {qvar} {current} -> {reduced}")
                    globals()[qvar] = reduced

    RAM_PEAK_GB, _, _, _, _ = _estimate_ram_peak_gb(globals())
    if RAM_PEAK_GB > RAM_BUDGET_GB:
        print(f"RAM guard result: still risky (~{RAM_PEAK_GB:.2f} GB > ~{RAM_BUDGET_GB:.2f} GB).")
    else:
        print(f"RAM guard result: OK (~{RAM_PEAK_GB:.2f} GB <= ~{RAM_BUDGET_GB:.2f} GB).")
else:
    print("RAM guard: OK (current selection fits estimated RAM budget).")


## 3) Model Download
Download the required model files (and optional components) into ComfyUI folders.


In [ ]:
# @title 3) Download models (Qwen-Image-Edit-2511 GGUF + Qwen2.5-VL GGUF + VAE + LoRA)
import os

COMFY = '/content/ComfyUI'

# Directories
UNET_DIR = f'{COMFY}/models/unet'
DIFF_DIR = f'{COMFY}/models/diffusion_models'
TE_DIR   = f'{COMFY}/models/text_encoders'
CLIP_DIR = f'{COMFY}/models/clip'
VAE_DIR  = f'{COMFY}/models/vae'
LORA_DIR = f'{COMFY}/models/loras'

for d in [UNET_DIR, DIFF_DIR, TE_DIR, CLIP_DIR, VAE_DIR, LORA_DIR]:
    os.makedirs(d, exist_ok=True)


def dl(url, outdir, fname):
    outpath = os.path.join(outdir, fname)
    marker = outpath + '.aria2'
    ready = os.path.exists(outpath) and os.path.getsize(outpath) > 0 and not os.path.exists(marker)
    if ready and outpath.lower().endswith('.gguf'):
        with open(outpath, 'rb') as stream:
            ready = stream.read(4) == b'GGUF'
    if ready:
        print('Already exists and validated:', outpath)
        return
    if os.path.exists(outpath) and not os.path.exists(marker):
        print('Removing invalid completed file:', outpath)
        os.remove(outpath)
    print('Downloading/resuming:', fname)

    hf_token = os.environ.get('HUGGINGFACE_TOKEN') or os.environ.get('HF_TOKEN')
    if hf_token and 'huggingface.co' in url:
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M --header="Authorization: Bearer {hf_token}" "{url}" -d "{outdir}" -o "{fname}"
        pass  # keep block valid for parsers
    else:
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d "{outdir}" -o "{fname}"
        pass  # keep block valid for parsers

# --- Qwen-Image-Edit-2511 GGUF (unsloth) ---
edit_fname = f"qwen-image-edit-2511-{QWEN_EDIT_QUANT}.gguf"
edit_url = f"https://huggingface.co/unsloth/Qwen-Image-Edit-2511-GGUF/resolve/main/{edit_fname}"
dl(edit_url, UNET_DIR, edit_fname)

edit_link = f"{DIFF_DIR}/{edit_fname}"
if not os.path.exists(edit_link):
    !ln -s "{UNET_DIR}/{edit_fname}" "{edit_link}"
    pass  # keep block valid

# --- Qwen2.5-VL-7B-Instruct GGUF (ggml-org) ---
vl_fname = f"Qwen2.5-VL-7B-Instruct-{QWEN_VL_QUANT}.gguf"
vl_url = f"https://huggingface.co/ggml-org/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/{vl_fname}"
dl(vl_url, TE_DIR, vl_fname)

vl_link = f"{CLIP_DIR}/{vl_fname}"
if not os.path.exists(vl_link):
    !ln -s "{TE_DIR}/{vl_fname}" "{vl_link}"
    pass  # keep block valid

# --- mmproj (required for image conditioning) ---
if DOWNLOAD_MMPROJ:
    mmproj_fname = f"mmproj-Qwen2.5-VL-7B-Instruct-{MMPROJ_QUANT}.gguf"
    mmproj_url = f"https://huggingface.co/ggml-org/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/{mmproj_fname}"
    dl(mmproj_url, TE_DIR, mmproj_fname)

    mmproj_link = f"{CLIP_DIR}/{mmproj_fname}"
    if not os.path.exists(mmproj_link):
        !ln -s "{TE_DIR}/{mmproj_fname}" "{mmproj_link}"
        pass  # keep block valid

# --- VAE ---
vae_fname = 'qwen_image_vae.safetensors'
vae_url = 'https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors'
dl(vae_url, VAE_DIR, vae_fname)

# --- LoRA (8 steps) ---
if DOWNLOAD_LORA:
    lora_fname = LORA_FILE
    lora_url = f"https://huggingface.co/lightx2v/Qwen-Image-Edit-2511-Lightning/resolve/main/{lora_fname}"
    dl(lora_url, LORA_DIR, lora_fname)

print('Done downloading models.')
print('Qwen-Edit :', edit_fname)
print('Qwen2.5-VL:', vl_fname)
print('VAE       :', vae_fname)
if DOWNLOAD_MMPROJ:
    print('mmproj    :', mmproj_fname)
if DOWNLOAD_LORA:
    print('LoRA      :', lora_fname)


# ---- Lenovo UltraReal LoRA pack (Civitai) ----
DOWNLOAD_LENOVO_ULTRAREAL_LORAS = True
CIVITAI_API_TOKEN = os.environ.get('CIVITAI_API_TOKEN', '').strip()  # optional, helps if Civitai returns 403.
CURRENT_BASE_MODELS = ['Qwen']

if DOWNLOAD_LENOVO_ULTRAREAL_LORAS:
    import json
    import subprocess
    import urllib.request
    import urllib.parse

    LORA_DIR = f'{COMFY}/models/loras'
    os.makedirs(LORA_DIR, exist_ok=True)

    def add_civitai_token(url, token):
        if not token:
            return url
        parts = urllib.parse.urlsplit(url)
        query = urllib.parse.parse_qsl(parts.query, keep_blank_values=True)
        if not any(k == 'token' for k, _ in query):
            query.append(('token', token))
        return urllib.parse.urlunsplit((parts.scheme, parts.netloc, parts.path, urllib.parse.urlencode(query), parts.fragment))

    def dl_civitai(url, outdir, fname, token=''):
        outpath = os.path.join(outdir, fname)
        if os.path.exists(outpath):
            print('Already exists:', outpath)
            return True

        final_url = add_civitai_token(url, token)
        print('Downloading LoRA:', fname)

        # 1) Fast path via aria2c
        cmd = ['aria2c', '--console-log-level=error', '-c', '-x', '8', '-s', '8', '-k', '1M', final_url, '-d', outdir, '-o', fname]
        rc = subprocess.run(cmd, check=False).returncode
        if rc == 0 and os.path.exists(outpath) and os.path.getsize(outpath) > 0:
            return True

        # 2) Fallback via urllib (some Civitai links fail in aria2 despite valid token)
        try:
            headers = {'User-Agent': 'Mozilla/5.0'}
            if token:
                headers['Authorization'] = f'Bearer {token}'
            req = urllib.request.Request(final_url, headers=headers)
            with urllib.request.urlopen(req, timeout=120) as resp, open(outpath, 'wb') as f:
                f.write(resp.read())
            if os.path.exists(outpath) and os.path.getsize(outpath) > 0:
                print('  Fallback downloader: OK')
                return True
        except Exception as e:
            print('  Fallback downloader failed:', e)

        if not token:
            print('  Download failed without token. Set CIVITAI_API_TOKEN env var and retry Cell 3.')
        else:
            print('  Download failed even with token. Check token validity and Civitai availability.')
        return False

    try:
        req = urllib.request.Request(
            'https://civitai.com/api/v1/models/1662740',
            headers={'User-Agent': 'Mozilla/5.0'}
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            civitai_model = json.loads(resp.read().decode('utf-8'))

        versions = civitai_model.get('modelVersions', [])
        print('\nLenovo UltraReal LoRA versions found:', len(versions))
        if not CURRENT_BASE_MODELS:
            print('Current notebook base-model tags: (none matched explicitly)')
        else:
            print('Current notebook base-model tags:', ', '.join(CURRENT_BASE_MODELS))

        for mv in versions:
            base = mv.get('baseModel', 'Unknown')
            match = base in CURRENT_BASE_MODELS
            mark = 'MATCH' if match else 'OTHER'
            files = [f for f in mv.get('files', []) if f.get('type') == 'Model']
            print(f"- [{mark}] {base} :: {mv.get('name', 'Unnamed version')} ({len(files)} file(s))")

        ok_count = 0
        total_count = 0
        matched_versions = [mv for mv in versions if mv.get('baseModel', 'Unknown') in CURRENT_BASE_MODELS]

        if not matched_versions:
            print('No MATCH versions found for this notebook base-model tag set. Skipping LoRA download.')

        for mv in matched_versions:
            for fobj in mv.get('files', []):
                if fobj.get('type') != 'Model':
                    continue
                total_count += 1
                raw_name = (fobj.get('name') or '').strip()
                base_tag = ''.join(ch if ch.isalnum() else '_' for ch in str(mv.get('baseModel', 'unknown'))).strip('_').lower() or 'unknown'
                fname = f"lenovo_ultrareal_{base_tag}_v{mv.get('id', 'x')}_f{fobj.get('id', 'x')}.safetensors"
                if raw_name and raw_name != fname:
                    print(f"  Civitai file name: {raw_name} -> saved as {fname}")
                if dl_civitai(fobj.get('downloadUrl', ''), LORA_DIR, fname, CIVITAI_API_TOKEN):
                    ok_count += 1

        print(f"\nLenovo UltraReal LoRA download summary (MATCH only): {ok_count}/{total_count} files ready in {LORA_DIR}")

    except Exception as e:
        print('Failed to fetch Lenovo UltraReal LoRA metadata from Civitai:', e)
        print('Tip: this may require CIVITAI_API_TOKEN or temporary Civitai availability.')


In [ ]:
# @title Install bundled workflow in ComfyUI
import json
from pathlib import Path
from urllib.request import urlopen

WORKFLOW_URL = "https://raw.githubusercontent.com/ekkonwork/free-comfyui-colab-pack/main/workflows/qwen_image_edit_2511/workflow.json"
WORKFLOW_DIR = Path("/content/ComfyUI/user/default/workflows")
WORKFLOW_PATH = WORKFLOW_DIR / "qwen_image_edit_2511.json"
WORKFLOW_DIR.mkdir(parents=True, exist_ok=True)
workflow = json.loads(urlopen(WORKFLOW_URL, timeout=60).read().decode("utf-8"))
replacements = {
    'qwen-image-edit-2511-Q2_K.gguf': edit_fname,
    'Qwen2.5-VL-7B-Instruct-Q4_K_M.gguf': vl_fname,
    'Qwen-Image-Edit-2511-Lightning-8steps-V1.0-bf16.safetensors': globals().get('lora_fname', 'Qwen-Image-Edit-2511-Lightning-8steps-V1.0-bf16.safetensors'),
}

def replace_runtime_names(value):
    if isinstance(value, str):
        return replacements.get(value, value)
    if isinstance(value, list):
        return [replace_runtime_names(item) for item in value]
    if isinstance(value, dict):
        return {key: replace_runtime_names(item) for key, item in value.items()}
    return value

workflow = replace_runtime_names(workflow)
WORKFLOW_PATH.write_text(json.dumps(workflow, ensure_ascii=False, indent=2), encoding="utf-8")
print("Bundled workflow installed:", WORKFLOW_PATH)


## 4) Download Check
Verify downloaded file sizes and calculate a rough VRAM estimate from real files.


In [ ]:
# @title 4) Verify downloads: actual file sizes and VRAM estimate
import os


def size_gb(path):
    return os.path.getsize(path) / (1024**3)


def est_vram_from_file(path, overhead=1.10):
    return size_gb(path) * overhead


paths = []
paths.append(('/content/ComfyUI/models/unet', f"qwen-image-edit-2511-{QWEN_EDIT_QUANT}.gguf"))
paths.append(('/content/ComfyUI/models/text_encoders', f"Qwen2.5-VL-7B-Instruct-{QWEN_VL_QUANT}.gguf"))
paths.append(('/content/ComfyUI/models/vae', 'qwen_image_vae.safetensors'))
if DOWNLOAD_MMPROJ:
    paths.append(('/content/ComfyUI/models/text_encoders', f"mmproj-Qwen2.5-VL-7B-Instruct-{MMPROJ_QUANT}.gguf"))
if DOWNLOAD_LORA:
    paths.append(('/content/ComfyUI/models/loras', LORA_FILE))

print('Files:')
total_vram = 0.0
for d, f in paths:
    p = os.path.join(d, f)
    if not os.path.exists(p):
        print('MISSING:', p)
        continue
    s = size_gb(p)
    v = est_vram_from_file(p)
    total_vram += v
    print(f"- {p}\n  size~{s:.2f} GB  -> VRAM(weights)~{v:.2f} GB")

print(f"\nSum VRAM(weights) if all kept on GPU at once: ~{total_vram:.2f} GB")
print('Reminder: real peak VRAM will be higher (resolution/batch/latents).')


## 5) Launch ComfyUI
Start ComfyUI and use the Cloudflare Tunnel URL shown in the output.


In [ ]:
# @title 5) Launch ComfyUI + Cloudflare Quick Tunnel
# Реализация туннеля идентична Hermes Dashboard-ячейке:
#   1. поднимаем ComfyUI локально и ждём готовности;
#   2. находим/скачиваем cloudflared;
#   3. останавливаем старый туннель при повторном запуске;
#   4. поднимаем Quick Tunnel на локальный порт ComfyUI;
#   5. читаем URL из лога, ждём DNS, проверяем публичный доступ.
import base64, os, re, shutil, socket, stat, subprocess, threading, time
import queue
from pathlib import Path

import requests

LOW_VRAM_STABLE = False  # False: --lowvram + Dynamic VRAM (optimal for T4 15GB); True: --novram ultra-low (2.5x slower, risks RAM OOM)
COMFY_ROOT = Path(globals().get("COMFY_ROOT", "/content/ComfyUI"))
OUTPUT_DIR = COMFY_ROOT / "output"
COMFY_PORT = globals().get("COMFY_PORT", 8188)

# ── Model Manager token bridge (keys live in its private.key pickle) ────────
MODEL_MANAGER_DIR = COMFY_ROOT / "custom_nodes" / "ComfyUI-Model-Manager"
if MODEL_MANAGER_DIR.exists():
    import pickle

    manager_key_file = MODEL_MANAGER_DIR / "private.key"
    manager_keys = {}
    if manager_key_file.exists():
        try:
            with manager_key_file.open("rb") as stream:
                loaded = pickle.load(stream)
            if isinstance(loaded, dict):
                manager_keys.update(loaded)
        except Exception:
            print("Existing Model Manager key file was unreadable; recreating it.")

    hf_for_manager = (os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN") or "").strip()
    civitai_for_manager = os.environ.get("CIVITAI_API_TOKEN", "").strip()
    if hf_for_manager:
        manager_keys["huggingface"] = hf_for_manager
    if civitai_for_manager:
        manager_keys["civitai"] = civitai_for_manager
    if manager_keys:
        manager_key_tmp = manager_key_file.with_suffix(".private.key.tmp")
        with manager_key_tmp.open("wb") as stream:
            pickle.dump(manager_keys, stream, protocol=pickle.HIGHEST_PROTOCOL)
        manager_key_tmp.replace(manager_key_file)
    print(
        "Model Manager token bridge:",
        "HF=" + ("yes" if hf_for_manager else "existing/none"),
        "Civitai=" + ("yes" if civitai_for_manager else "existing/none"),
    )
else:
    print("⚠ ComfyUI-Model-Manager not found — run the install cell first.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def stop_process(proc):
    if proc is None or proc.poll() is not None:
        return
    proc.terminate()
    try:
        proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proc.kill()


# Safe rerun: stop processes started by any earlier launch-cell version.
for process_name in ("_TUNNEL_PROC", "_COMFY_PROC"):
    stop_process(globals().get(process_name))
old_log = globals().get("_COMFY_LOG")
if old_log is not None:
    try:
        old_log.close()
    except Exception:
        pass


def port_open(host, port, timeout=1):
    s = socket.socket()
    s.settimeout(timeout)
    try:
        return s.connect_ex((host, port)) == 0
    finally:
        s.close()


# ══════════════════════════════════════════════════════════════════════════
# 1. ЗАПУСК COMFYUI ЛОКАЛЬНО
# ══════════════════════════════════════════════════════════════════════════
comfy_args = [
    "python", "main.py", "--listen", "0.0.0.0", "--port", str(COMFY_PORT),
    "--enable-cors-header", "*", "--output-directory", str(OUTPUT_DIR),
]
comfy_args += (
    ["--novram", "--disable-smart-memory", "--cache-none", "--force-upcast-attention"]
    if LOW_VRAM_STABLE else ["--lowvram", "--preview-method", "auto"]
)
_COMFY_LOG = open("/content/comfyui.log", "a", encoding="utf-8", buffering=1)
_COMFY_PROC = subprocess.Popen(
    comfy_args, cwd=COMFY_ROOT, stdout=_COMFY_LOG, stderr=subprocess.STDOUT,
)


def local_comfy_ready(timeout=300):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if _COMFY_PROC.poll() is not None:
            raise RuntimeError("ComfyUI exited. Inspect /content/comfyui.log")
        try:
            response = requests.get(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=5)
            if response.ok:
                return True
        except requests.RequestException:
            time.sleep(2)
    return False


if not local_comfy_ready():
    raise TimeoutError("ComfyUI did not become ready within 300 seconds.")
print("✓ ComfyUI is ready locally on port", COMFY_PORT)


# ══════════════════════════════════════════════════════════════════════════
# 2. НАХОДИМ / СКАЧИВАЕМ CLOUDFLARED  (как в Hermes Dashboard-ячейке)
# ══════════════════════════════════════════════════════════════════════════
cloudflared_candidates = [
    shutil.which("cloudflared"),
    "/usr/local/bin/cloudflared",
]

CLOUDFLARED_BIN = None

for c in cloudflared_candidates:
    if c and Path(c).exists():
        CLOUDFLARED_BIN = Path(c)
        break

if CLOUDFLARED_BIN is None:
    machine = os.uname().machine.lower()

    if machine in ("x86_64", "amd64"):
        cf_arch = "amd64"
    elif machine in ("aarch64", "arm64"):
        cf_arch = "arm64"
    else:
        raise RuntimeError(f"Unknown architecture: {machine}")

    CLOUDFLARED_BIN = Path("/tmp/cloudflared-comfy")
    download_url = (
        "https://github.com/cloudflare/cloudflared/"
        f"releases/latest/download/cloudflared-linux-{cf_arch}"
    )

    print("Скачиваю cloudflared...")
    subprocess.run(
        ["curl", "-fL", "--retry", "3", download_url, "-o", str(CLOUDFLARED_BIN)],
        check=True,
    )
    CLOUDFLARED_BIN.chmod(CLOUDFLARED_BIN.stat().st_mode | stat.S_IXUSR)

print("cloudflared:", CLOUDFLARED_BIN)


# ══════════════════════════════════════════════════════════════════════════
# 3. ПОВТОРНЫЙ ЗАПУСК — ОСТАНАВЛИВАЕМ СТАРЫЙ ТУННЕЛЬ
# ══════════════════════════════════════════════════════════════════════════
old_cf = globals().get("_CLOUDFLARED_PROC")
if old_cf is not None:
    try:
        if old_cf.poll() is None:
            print("Останавливаю предыдущий tunnel...")
            old_cf.terminate()
            try:
                old_cf.wait(timeout=10)
            except Exception:
                old_cf.kill()
    except Exception:
        pass


# ══════════════════════════════════════════════════════════════════════════
# 4. ПОДНИМАЕМ QUICK TUNNEL НА COMFYUI
# ══════════════════════════════════════════════════════════════════════════
print("=" * 78)
print("ЗАПУСК CLOUDFLARE QUICK TUNNEL -> COMFYUI")
print("=" * 78)

_CLOUDFLARED_PROC = subprocess.Popen(
    [
        str(CLOUDFLARED_BIN),
        "tunnel",
        "--no-autoupdate",
        "--url",
        f"http://127.0.0.1:{COMFY_PORT}",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)


# ══════════════════════════════════════════════════════════════════════════
# 5. ЧИТАЕМ URL ИЗ ЛОГА
# ══════════════════════════════════════════════════════════════════════════
TUNNEL_URL = None

cf_lines = []

pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")

deadline = time.time() + 90

while time.time() < deadline:

    if _CLOUDFLARED_PROC.poll() is not None:
        break

    line = _CLOUDFLARED_PROC.stdout.readline()

    if line:
        cf_lines.append(line.rstrip())
        print(line, end="")

        match = pattern.search(line)

        if match:
            TUNNEL_URL = match.group(0)
            break

    else:
        time.sleep(0.2)

if not TUNNEL_URL:
    print()
    print("=== CLOUDFLARED OUTPUT ===")
    print("\n".join(cf_lines[-200:]))

    # Fallback hint: the next cell exposes ComfyUI through bore.pub instead.
    raise RuntimeError(
        "Не удалось получить Quick Tunnel URL. "
        "Если сеть блокирует trycloudflare — запусти следующую ячейку (bore.pub fallback)."
    )

print()
print("✓ ComfyUI Tunnel:", TUNNEL_URL)


# ══════════════════════════════════════════════════════════════════════════
# 6. ЖДЁМ DNS
# ══════════════════════════════════════════════════════════════════════════
TUNNEL_HOST = TUNNEL_URL.replace("https://", "").split("/")[0]

dns_ok = False

for attempt in range(1, 31):
    try:
        infos = socket.getaddrinfo(TUNNEL_HOST, 443)
        if infos:
            dns_ok = True
            break
    except socket.gaierror:
        pass
    time.sleep(2)

print("DNS:", "OK" if dns_ok else "NOT READY")


# ══════════════════════════════════════════════════════════════════════════
# 7. ПРОВЕРЯЕМ ПУБЛИЧНЫЙ ДОСТУП
# ══════════════════════════════════════════════════════════════════════════
if dns_ok:

    print("=" * 78)
    print("PUBLIC COMFYUI TEST")
    print("=" * 78)

    public_test = subprocess.run(
        [
            "curl", "-sS", "-L",
            "--connect-timeout", "15",
            "--max-time", "60",
            "--retry", "5",
            "--retry-delay", "2",
            "-o", "/dev/null",
            "-w", "%{http_code}",
            TUNNEL_URL + "/system_stats",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        timeout=90,
    )

    public_http = public_test.stdout.strip()
    print("Public /system_stats HTTP:", public_http)

    if public_test.stderr:
        print("curl stderr:", public_test.stderr[:3000])

    if public_http not in ("200", "403"):
        print("⚠ Public check inconclusive — tunnel может ещё прогреваться.")

print()
print("=" * 78)
print("✓ COMFYUI ГОТОВ — ОТКРЫВАЙ В БРАУЗЕРЕ:")
print(TUNNEL_URL)
print("=" * 78)
print("Если ссылка даёт 403 — запусти следующую ячейку (bore.pub fallback).")

globals()["TUNNEL_URL"] = TUNNEL_URL

In [ ]:
# @title 5a) Watchdog: ComfyUI + Cloudflare Tunnel keepalive
# Keeps BOTH services alive. ComfyUI-Manager reboot is given a grace period:
# if Manager already spawned/replaced main.py, watchdog detects that process/port
# and never starts a duplicate.
import os, re, shutil, socket, subprocess, time
from pathlib import Path
import requests

COMFY_PORT = int(globals().get("COMFY_PORT", 8188))
COMFY_ROOT = Path(globals().get("COMFY_ROOT", "/content/ComfyUI"))
COMFY_RESTART_GRACE = 25
COMFY_HUNG_GRACE = 120
CLOUDFLARED_BIN = globals().get("CLOUDFLARED_BIN")
if CLOUDFLARED_BIN is None:
    for candidate in [shutil.which("cloudflared"), "/usr/local/bin/cloudflared", "/tmp/cloudflared-comfy"]:
        if candidate and Path(candidate).exists():
            CLOUDFLARED_BIN = Path(candidate)
            break
if CLOUDFLARED_BIN is None:
    raise RuntimeError("cloudflared not found — run the launch/tunnel cell first.")

_cf_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")

def _wd_health():
    try:
        return requests.get(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=4).ok
    except Exception:
        return False

def _wd_main_pids():
    # Detect a Manager-started replacement before spawning our own process.
    try:
        out = subprocess.check_output(
            ["pgrep", "-f", rf"python.*main\.py.*--port[ =]{COMFY_PORT}"],
            text=True, stderr=subprocess.DEVNULL,
        )
        return [int(x) for x in out.split() if x.isdigit()]
    except Exception:
        return []

def _wd_comfy_args():
    args = globals().get("comfy_args")
    if isinstance(args, (list, tuple)) and args:
        return list(args)
    output_dir = COMFY_ROOT / "output"
    low_stable = bool(globals().get("LOW_VRAM_STABLE", False))
    args = [
        "python", "main.py", "--listen", "0.0.0.0", "--port", str(COMFY_PORT),
        "--enable-cors-header", "*", "--output-directory", str(output_dir),
    ]
    args += (
        ["--novram", "--disable-smart-memory", "--cache-none", "--force-upcast-attention"]
        if low_stable else ["--lowvram", "--preview-method", "auto"]
    )
    globals()["comfy_args"] = args
    return args

def _wd_spawn_comfy():
    # Final race check: Manager may have completed reboot while we were waiting.
    if _wd_health() or _wd_main_pids():
        return False
    old_log = globals().get("_COMFY_LOG")
    try:
        if old_log is not None:
            old_log.close()
    except Exception:
        pass
    log = open("/content/comfyui.log", "a", encoding="utf-8", buffering=1)
    proc = subprocess.Popen(
        _wd_comfy_args(), cwd=COMFY_ROOT, stdout=log, stderr=subprocess.STDOUT
    )
    globals()["_COMFY_LOG"] = log
    globals()["_COMFY_PROC"] = proc
    globals()["_COMFY_DOWN_SINCE"] = time.time()
    print(f"[{time.strftime('%H:%M:%S')}] ComfyUI watchdog spawned PID {proc.pid}.")
    return True

def _ensure_comfy_running():
    if _wd_health():
        globals()["_COMFY_DOWN_SINCE"] = None
        return True

    now = time.time()
    down_since = globals().get("_COMFY_DOWN_SINCE")
    if not down_since:
        down_since = now
        globals()["_COMFY_DOWN_SINCE"] = down_since

    proc = globals().get("_COMFY_PROC")
    proc_alive = proc is not None and proc.poll() is None
    pids = _wd_main_pids()

    # Most important Manager-compat rule: if any replacement main.py exists,
    # do not race it. The same Cloudflare tunnel already targets localhost:port.
    if pids and (not proc_alive or proc.pid not in pids):
        if int(now - down_since) % 30 < 5:
            print(f"[{time.strftime('%H:%M:%S')}] Manager/external ComfyUI startup detected (PIDs {pids}); waiting for port.")
        return False

    grace = COMFY_HUNG_GRACE if proc_alive else COMFY_RESTART_GRACE
    if now - down_since < grace:
        return False

    if proc_alive:
        print(f"[{time.strftime('%H:%M:%S')}] ComfyUI owned PID {proc.pid} is unhealthy for {grace}s; restarting it.")
        try:
            proc.terminate()
            proc.wait(timeout=10)
        except Exception:
            try:
                proc.kill()
            except Exception:
                pass
        time.sleep(2)

    if _wd_health() or _wd_main_pids():
        return _wd_health()
    _wd_spawn_comfy()

    deadline = time.time() + 300
    while time.time() < deadline:
        if _wd_health():
            globals()["_COMFY_DOWN_SINCE"] = None
            print(f"[{time.strftime('%H:%M:%S')}] ✓ ComfyUI recovered; existing tunnel remains attached to port {COMFY_PORT}.")
            return True
        proc = globals().get("_COMFY_PROC")
        if proc is not None and proc.poll() is not None:
            print(f"[{time.strftime('%H:%M:%S')}] ComfyUI restart exited={proc.poll()}; retry will happen on next watchdog cycle.")
            break
        time.sleep(3)
    return False

def _restart_cf():
    old = globals().get("_CLOUDFLARED_PROC")
    if old is not None and old.poll() is None:
        try:
            old.terminate()
            old.wait(timeout=5)
        except Exception:
            try: old.kill()
            except Exception: pass
    print(f"[{time.strftime('%H:%M:%S')}] Restarting cloudflared -> 127.0.0.1:{COMFY_PORT} ...")
    proc = subprocess.Popen(
        [str(CLOUDFLARED_BIN), "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{COMFY_PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    globals()["_CLOUDFLARED_PROC"] = proc
    url = None
    deadline = time.time() + 90
    lines = []
    while time.time() < deadline:
        if proc.poll() is not None:
            break
        line = proc.stdout.readline() if proc.stdout else ""
        if line:
            lines.append(line.rstrip())
            print(line, end="")
            match = _cf_pattern.search(line)
            if match:
                url = match.group(0)
                break
        else:
            time.sleep(0.2)
    if not url:
        print("\n".join(lines[-20:]) if lines else "cloudflared produced no URL")
        return None
    globals()["TUNNEL_URL"] = url
    print(f"✓ TUNNEL_URL: {url}")
    return url

_ensure_comfy_running()
cf = globals().get("_CLOUDFLARED_PROC")
if cf is None or cf.poll() is not None:
    _restart_cf()

print("Unified ComfyUI + Cloudflare watchdog started. Interrupt this cell to stop monitoring.")
_tick = 0
try:
    while True:
        time.sleep(5)
        _tick += 5
        local_ok = _ensure_comfy_running()
        cf = globals().get("_CLOUDFLARED_PROC")
        if cf is None or cf.poll() is not None:
            _restart_cf()
            _tick = 0
            continue
        if _tick >= 60:
            _tick = 0
            url = globals().get("TUNNEL_URL", "")
            if local_ok and url:
                try:
                    public = requests.get(url + "/system_stats", timeout=8)
                    print(f"[{time.strftime('%H:%M:%S')}] keepalive local=OK public={public.status_code} {url}")
                    if public.status_code >= 500:
                        _restart_cf()
                except Exception as exc:
                    print(f"[{time.strftime('%H:%M:%S')}] public ping failed ({exc}); restarting tunnel only.")
                    _restart_cf()
            else:
                print(f"[{time.strftime('%H:%M:%S')}] keepalive local={'OK' if local_ok else 'RECOVERING'}")
except KeyboardInterrupt:
    print("Watchdog stopped by user.")


In [ ]:
# @title 5b) Fallback: bore.pub tunnel (если Cloudflare URL заблокирован / 403)
# Некоторые сети/страны получают 403 от Cloudflare на *.trycloudflare.com ссылки.
# Эта ячейка открывает ТОТ ЖЕ локальный ComfyUI через bore.pub.
# Обычный HTTP без шифрования и случайный порт — используй только как fallback.
# Останавливает CF-туннель, чтобы не держать два сразу.
import os
import re as _re
import subprocess
import threading as _threading
import time

import requests

COMFY_PORT = globals().get("COMFY_PORT", 8188)

# Останавливаем Cloudflare туннель из предыдущей ячейки (если был).
_old_cf = globals().get("_CLOUDFLARED_PROC")
if _old_cf is not None and _old_cf.poll() is None:
    _old_cf.terminate()
    try:
        _old_cf.wait(timeout=10)
    except Exception:
        _old_cf.kill()
    print("Cloudflare tunnel остановлен.")


def _ensure_bore():
    import shutil

    if shutil.which("bore"):
        return "bore"
    local = "/tmp/bore"
    if os.path.exists(local) and os.access(local, os.X_OK):
        return local
    arch = {"x86_64": "x86_64", "amd64": "x86_64", "aarch64": "aarch64"}.get(
        os.uname().machine.lower(), "x86_64"
    )
    url = (
        "https://github.com/ekzhang/bore/releases/download/v0.5.0/"
        f"bore-v0.5.0-{arch}-unknown-linux-musl.tar.gz"
    )
    print("Скачиваю bore CLI...")
    subprocess.run(["curl", "-fsSL", "-o", "/tmp/bore.tar.gz", url], check=True)
    subprocess.run(["tar", "-xzf", "/tmp/bore.tar.gz", "-C", "/tmp"], check=True)
    os.chmod("/tmp/bore", 0o755)
    return local


BORE_BIN = _ensure_bore()

# Повторный запуск ячейки — останавливаем старый fallback-туннель.
old = globals().get("_BORE_PROC")
if old is not None and old.poll() is None:
    old.terminate()
    try:
        old.wait(timeout=5)
    except Exception:
        old.kill()


# Убеждаемся, что ComfyUI ещё жив (запущен предыдущей ячейкой).
assert globals().get("_COMFY_PROC") is not None and _COMFY_PROC.poll() is None, (
    "ComfyUI не запущен — сначала выполни предыдущую ячейку."
)

_BORE_LOG = open("/content/bore.log", "a", encoding="utf-8", buffering=1)
_BORE_PROC = subprocess.Popen(
    [BORE_BIN, "local", str(COMFY_PORT), "--to", "bore.pub"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

_public_port = None
_deadline = time.time() + 45
_lines = []


def _pump(pipe):
    # Robust: pipe may be None (if Popen failed) or closed; avoid AttributeError
    if pipe is None:
        _lines.append("[bore] no stdout pipe (Popen failed)")
        return
    try:
        for line in iter(pipe.readline, ""):
            if line is None:
                break
            _lines.append(line.rstrip())
            if not line:
                time.sleep(0.1)
    except Exception as e:
        _lines.append(f"[bore pump error] {e}")


_threading.Thread(target=_pump, args=(_BORE_PROC.stdout,), daemon=True).start()

while time.time() < _deadline and _public_port is None:
    if _BORE_PROC.poll() is not None:
        print("bore процесс завершился преждевременно. Лог:")
        print("\n".join(_lines[-20:]))
        break
    for line in list(_lines):
        m = _re.search(r"listening at bore\.pub:(\d+)", line)
        if m:
            _public_port = m.group(1)
            break
    time.sleep(0.5)

if not _public_port:
    print("\n".join(_lines[-20:]) if _lines else "(нет вывода от bore)")
    raise RuntimeError("bore.pub tunnel failed to start. Попробуй перезапустить ячейку или используй Cloudflare tunnel (предыдущая ячейка).")

PUBLIC_URL = f"http://bore.pub:{_public_port}"
print("=" * 78)
print("✓ COMFYUI ЧЕРЕЗ BORE.PUB:", PUBLIC_URL)
print("(plain HTTP — используй только если Cloudflare URL заблокирован)")
print("=" * 78)

for _ in range(6):
    try:
        if requests.get(PUBLIC_URL + "/system_stats", timeout=8).ok:
            print("Public check: OK")
            break
    except requests.RequestException:
        pass
    time.sleep(3)

globals()["PUBLIC_URL"] = PUBLIC_URL


In [ ]:
# @title 5c) Watchdog: ComfyUI + bore.pub keepalive
# Same ComfyUI supervision rules as Cloudflare watchdog; safe with ComfyUI-Manager reboot.
import os, re, subprocess, time
from pathlib import Path
import requests

COMFY_PORT = int(globals().get("COMFY_PORT", 8188))
COMFY_ROOT = Path(globals().get("COMFY_ROOT", "/content/ComfyUI"))
COMFY_RESTART_GRACE = 25
COMFY_HUNG_GRACE = 120
BORE_BIN = globals().get("BORE_BIN")
if BORE_BIN is None:
    import shutil
    BORE_BIN = shutil.which("bore") or ("/tmp/bore" if Path("/tmp/bore").exists() else None)
if BORE_BIN is None:
    raise RuntimeError("bore not found — run the bore fallback cell first.")

def _wd_health():
    try:
        return requests.get(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=4).ok
    except Exception:
        return False

def _wd_main_pids():
    try:
        out = subprocess.check_output(["pgrep", "-f", rf"python.*main\.py.*--port[ =]{COMFY_PORT}"], text=True, stderr=subprocess.DEVNULL)
        return [int(x) for x in out.split() if x.isdigit()]
    except Exception:
        return []

def _wd_comfy_args():
    args = globals().get("comfy_args")
    if isinstance(args, (list, tuple)) and args:
        return list(args)
    output_dir = COMFY_ROOT / "output"
    low_stable = bool(globals().get("LOW_VRAM_STABLE", False))
    args = ["python", "main.py", "--listen", "0.0.0.0", "--port", str(COMFY_PORT), "--enable-cors-header", "*", "--output-directory", str(output_dir)]
    args += (["--novram", "--disable-smart-memory", "--cache-none", "--force-upcast-attention"] if low_stable else ["--lowvram", "--preview-method", "auto"])
    globals()["comfy_args"] = args
    return args

def _ensure_comfy_running():
    if _wd_health():
        globals()["_COMFY_DOWN_SINCE"] = None
        return True
    now = time.time()
    down = globals().get("_COMFY_DOWN_SINCE") or now
    globals()["_COMFY_DOWN_SINCE"] = down
    proc = globals().get("_COMFY_PROC")
    alive = proc is not None and proc.poll() is None
    pids = _wd_main_pids()
    if pids and (not alive or proc.pid not in pids):
        return False
    grace = COMFY_HUNG_GRACE if alive else COMFY_RESTART_GRACE
    if now - down < grace:
        return False
    if alive:
        try:
            proc.terminate(); proc.wait(timeout=10)
        except Exception:
            try: proc.kill()
            except Exception: pass
        time.sleep(2)
    if _wd_health() or _wd_main_pids():
        return _wd_health()
    log = open("/content/comfyui.log", "a", encoding="utf-8", buffering=1)
    proc = subprocess.Popen(_wd_comfy_args(), cwd=COMFY_ROOT, stdout=log, stderr=subprocess.STDOUT)
    globals()["_COMFY_LOG"] = log
    globals()["_COMFY_PROC"] = proc
    print(f"[{time.strftime('%H:%M:%S')}] ComfyUI watchdog spawned PID {proc.pid}.")
    deadline = time.time() + 300
    while time.time() < deadline:
        if _wd_health():
            globals()["_COMFY_DOWN_SINCE"] = None
            print("✓ ComfyUI recovered; bore remains attached to the same localhost port.")
            return True
        if proc.poll() is not None:
            break
        time.sleep(3)
    return False

def _restart_bore():
    old = globals().get("_BORE_PROC")
    if old is not None and old.poll() is None:
        try:
            old.terminate(); old.wait(timeout=5)
        except Exception:
            try: old.kill()
            except Exception: pass
    proc = subprocess.Popen([str(BORE_BIN), "local", str(COMFY_PORT), "--to", "bore.pub"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    globals()["_BORE_PROC"] = proc
    lines = []
    deadline = time.time() + 45
    port = None
    while time.time() < deadline and port is None:
        if proc.poll() is not None:
            break
        line = proc.stdout.readline() if proc.stdout else ""
        if line:
            lines.append(line.rstrip())
            m = re.search(r"listening at bore\.pub:(\d+)", line)
            if m: port = m.group(1)
        else:
            time.sleep(0.2)
    if not port:
        print("\n".join(lines[-20:]) if lines else "bore produced no public port")
        return None
    url = f"http://bore.pub:{port}"
    globals()["PUBLIC_URL"] = url
    print("✓ PUBLIC_URL:", url)
    return url

_ensure_comfy_running()
bore = globals().get("_BORE_PROC")
if bore is None or bore.poll() is not None:
    _restart_bore()

print("Unified ComfyUI + bore watchdog started. Interrupt this cell to stop monitoring.")
_tick = 0
try:
    while True:
        time.sleep(5)
        _tick += 5
        local_ok = _ensure_comfy_running()
        bore = globals().get("_BORE_PROC")
        if bore is None or bore.poll() is not None:
            _restart_bore(); _tick = 0; continue
        if _tick >= 60:
            _tick = 0
            url = globals().get("PUBLIC_URL", "")
            if local_ok and url:
                try:
                    public = requests.get(url + "/system_stats", timeout=8)
                    print(f"[{time.strftime('%H:%M:%S')}] keepalive local=OK public={public.status_code} {url}")
                except Exception as exc:
                    print(f"[{time.strftime('%H:%M:%S')}] bore public ping failed ({exc}); restarting tunnel only.")
                    _restart_bore()
            else:
                print(f"[{time.strftime('%H:%M:%S')}] keepalive local={'OK' if local_ok else 'RECOVERING'}")
except KeyboardInterrupt:
    print("Watchdog stopped by user.")
